# 🌱 LiFE Pulse — Notebook 1: Household kWh Estimator (M1)
### Mission LiFE Citizen Engagement App (HC26-GEDA01)

This notebook generates 10,000 calibrated Indian urban household data samples, trains an **XGBoost / Gradient Boosting Regressor** with SHAP explainability, and exports the model artifact `kwh_model.joblib`.

---

### Step 1: Mount Google Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = '/content/drive/MyDrive/lifepulse'
os.makedirs(f'{BASE_DIR}/artifacts', exist_ok=True)
os.makedirs(f'{BASE_DIR}/data', exist_ok=True)

!pip install -q xgboost scikit-learn pandas numpy joblib shap matplotlib

### Step 2: Generate Calibrated Synthetic Dataset (10,000 Indian Households)
Calibrated against Central Electricity Authority (CEA) baselines and Gujarat urban appliance habits.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
N = 10000

df = pd.DataFrame({
    'members': rng.integers(1, 8, N),
    'home_type': rng.choice([0, 1, 2, 3], N, p=[.25, .40, .25, .10]),   # 1BHK, 2BHK, 3BHK, Bungalow
    'ac_count': rng.choice([0, 1, 2, 3], N, p=[.35, .35, .20, .10]),
    'ac_star': rng.integers(1, 6, N),
    'ac_hours': rng.uniform(0, 10, N),
    'fridge': rng.choice([0, 1, 2], N, p=[.05, .85, .10]),
    'geyser': rng.choice([0, 1], N, p=[.40, .60]),
    'geyser_hours': rng.uniform(0.2, 2.0, N),
    'fans': rng.integers(1, 8, N),
    'led_share': rng.uniform(0, 1.0, N),
    'lights': rng.integers(4, 25, N),
    'washing': rng.choice([0, 1], N, p=[.30, .70]),
    'ev': rng.choice([0, 1], N, p=[.93, .07]),
    'season': rng.integers(0, 3, N)   # 0: Winter, 1: Summer, 2: Monsoon
})

# Physics-calibrated electricity consumption formula + 8% stochastic noise
kwh = (
    df.ac_count * df.ac_hours * 1.4 * 30 * (1.2 - df.ac_star * 0.05) * np.where(df.season == 1, 1.3, 0.6)
    + df.geyser * df.geyser_hours * 2.0 * 30 * np.where(df.season == 0, 1.2, 0.5)
    + df.fridge * 45.0
    + df.fans * 0.075 * 10 * 30
    + df.lights * (df.led_share * 9 + (1 - df.led_share) * 40) / 1000 * 5 * 30
    + df.washing * 8.0
    + df.ev * 90.0
    + df.members * 14.0
)
df['kwh_month'] = (kwh * rng.normal(1.0, 0.08, N)).clip(35, 1400)

csv_path = f'{BASE_DIR}/data/synthetic_households.csv'
df.to_csv(csv_path, index=False)
print(f'✅ Dataset generated: {len(df)} rows saved to {csv_path}')
df.head()

### Step 3: Train XGBoost Regressor

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor
import joblib, json

X = df.drop(columns=['kwh_month'])
y = df['kwh_month']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('🚀 Training XGBoost Regressor (400 estimators)...')
model = XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

print(f'\n🎯 Test Evaluation Results:')
print(f'  • Mean Absolute Error (MAE): {mae:.2f} kWh/month')
print(f'  • R² Score: {r2:.4f}')

# Export artifacts to Drive
artifact_model_path = f'{BASE_DIR}/artifacts/kwh_model.joblib'
joblib.dump(model, artifact_model_path)

with open(f'{BASE_DIR}/artifacts/feature_order.json', 'w') as f:
    json.dump(list(X.columns), f, indent=2)

metrics = {'mae': float(mae), 'r2': float(r2), 'n_samples': N}
with open(f'{BASE_DIR}/artifacts/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'💾 Model saved to: {artifact_model_path}')

### Step 4: SHAP Feature Importance Plot (Pitch Deck Explainability)

In [ ]:
import shap
import matplotlib.pyplot as plt

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test[:500])

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test[:500], show=True)